# Tenacious Bench - ORPO (No Unsloth)

Backbone: `Qwen/Qwen2.5-0.5B-Instruct`  
Method: ORPO with LoRA (`transformers + peft + trl`)  
Includes: training log, ablation scaffold, artifact export


In [ ]:
# Cell 1 - Install
!pip uninstall -y unsloth unsloth_zoo trl transformers peft accelerate bitsandbytes tokenizers datasets huggingface-hub xformers triton torchao
!pip install --no-cache-dir \
  "torch==2.5.1" \
  "torchvision==0.20.1" \
  "torchaudio==2.5.1" \
  "triton==3.1.0" \
  "transformers==4.46.3" \
  "trl==0.15.2" \
  "peft==0.14.0" \
  "accelerate==0.34.2" \
  "bitsandbytes==0.45.5" \
  "datasets==3.6.0" \
  "tokenizers==0.20.3" \
  "huggingface-hub==0.36.2" \
  "protobuf>=5.29.1,<6" \
  "scipy" "statsmodels"
print("Install complete. Restart runtime/session now, then continue.")


In [ ]:
# Cell 2 - Config
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    max_seq_length: int = 1024
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    orpo_lambda: float = 0.1
    learning_rate: float = 8e-5
    num_epochs: int = 3
    batch_size: int = 2
    grad_accum: int = 4
    warmup_ratio: float = 0.1
    lr_scheduler: str = "cosine"
    seed: int = 42
    output_dir: str = "/content/tenacious-judge-orpo"
    hub_model_id: str = "YOUR_USERNAME/tenacious-judge-orpo"
    train_data_path: str = "/content/training_data/simpo_train.jsonl"
    eval_data_path: str = "/content/training_data/simpo_eval.jsonl"

cfg = Config()
cfg


In [ ]:
# Cell 3 - Load model + tokenizer + LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.target_modules,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 4 - Load dataset
from datasets import load_dataset
from collections import Counter
import statistics

train_ds = load_dataset("json", data_files=cfg.train_data_path, split="train")
eval_ds = load_dataset("json", data_files=cfg.eval_data_path, split="train")
print("Train:", len(train_ds), "Eval:", len(eval_ds))
print("Columns:", train_ds.column_names)

if "dimension" in train_ds.column_names:
    print("\nDimension distribution:")
    for dim, n in Counter(train_ds["dimension"]).most_common():
        print(f"  {dim}: {n}")

chosen_lens = [len(tokenizer(x["chosen"])["input_ids"]) for x in train_ds]
rejected_lens = [len(tokenizer(x["rejected"])["input_ids"]) for x in train_ds]
print(f"\nChosen mean/max: {statistics.mean(chosen_lens):.1f}/{max(chosen_lens)}")
print(f"Rejected mean/max: {statistics.mean(rejected_lens):.1f}/{max(rejected_lens)}")


In [ ]:
# Cell 5 - ORPO training
import inspect
from trl import ORPOConfig, ORPOTrainer

orpo_kwargs = dict(
    output_dir=cfg.output_dir,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type=cfg.lr_scheduler,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    save_total_limit=2,
    seed=cfg.seed,
    data_seed=cfg.seed,
    fp16=True,
    bf16=False,
    max_length=cfg.max_seq_length,
    max_prompt_length=768,
    report_to="none",
    run_name=f"tenacious-orpo-r{cfg.lora_r}-lam{cfg.orpo_lambda}",
)

sig = inspect.signature(ORPOConfig.__init__).parameters
if "lambda_" in sig:
    orpo_kwargs["lambda_"] = cfg.orpo_lambda
elif "orpo_alpha" in sig:
    orpo_kwargs["orpo_alpha"] = cfg.orpo_lambda

orpo_config = ORPOConfig(**orpo_kwargs)
trainer_kwargs = dict(
    model=model,
    args=orpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer_sig = inspect.signature(ORPOTrainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = ORPOTrainer(**trainer_kwargs)
train_result = trainer.train()
print(f"Done. Final loss: {train_result.training_loss:.4f}")


In [ ]:
# Cell 6 - Save training log
import json
from pathlib import Path

log_history = trainer.state.log_history
train_log = [e for e in log_history if "loss" in e and "eval_loss" not in e]
eval_log = [e for e in log_history if "eval_loss" in e]

training_run = {
    "algorithm": "ORPO",
    "hyperparameters": {
        "backbone": cfg.model_name,
        "lora_r": cfg.lora_r,
        "lora_alpha": cfg.lora_alpha,
        "lora_dropout": cfg.lora_dropout,
        "target_modules": cfg.target_modules,
        "orpo_lambda": cfg.orpo_lambda,
        "learning_rate": cfg.learning_rate,
        "num_epochs": cfg.num_epochs,
        "batch_size": cfg.batch_size,
        "grad_accum": cfg.grad_accum,
        "seed": cfg.seed,
        "max_seq_length": cfg.max_seq_length
    },
    "training_data": {
        "train_pairs": len(train_ds),
        "eval_pairs": len(eval_ds)
    },
    "results": {
        "final_train_loss": train_log[-1].get("loss") if train_log else None,
        "final_eval_loss": eval_log[-1].get("eval_loss") if eval_log else None,
        "training_steps": train_log[-1].get("step") if train_log else None
    },
    "loss_history": {
        "train": [(e.get("step"), e.get("loss")) for e in train_log],
        "eval": [(e.get("step"), e.get("eval_loss")) for e in eval_log]
    }
}
Path("/content/training_run_orpo.log").write_text(json.dumps(training_run, indent=2))
print("Saved: /content/training_run_orpo.log")


In [ ]:
# Cell 7 - Save adapter + tokenizer (optional push)
import os
os.makedirs(cfg.output_dir, exist_ok=True)
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print("Saved:", cfg.output_dir)

# Optional push
# from huggingface_hub import login
# hf_token = os.environ.get("HF_TOKEN", "")
# login(token=hf_token)
# model.push_to_hub(cfg.hub_model_id, token=hf_token)
# tokenizer.push_to_hub(cfg.hub_model_id, token=hf_token)


In [ ]:
# Cell 8 - Ablation scaffold
import json
import sys
import numpy as np
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar

held_out_path = "/content/training_data/held_out_tasks.jsonl.jsonl"
held_out_tasks = [json.loads(x) for x in Path(held_out_path).read_text().splitlines() if x.strip()]
print("Held-out tasks:", len(held_out_tasks))

# Ensure scoring_evaluator.py exists in /content or adjust path
sys.path.insert(0, "/content")
from scoring_evaluator import score_task

THRESH = 0.65

def baseline(t):
    return t.get("rejected", "")

def mechanism(t):
    draft = baseline(t)
    return draft if score_task(t, draft)["pct"] >= THRESH else t.get("chosen", draft)

def rule_judge(t):
    draft = baseline(t)
    banned = ["aggressively", "tripling", "we can provide go", "world-class", "top talent"]
    return t.get("chosen", draft) if any(b in draft.lower() for b in banned) else draft

def pass_vec(fn):
    return [1 if score_task(t, fn(t))["pass"] else 0 for t in held_out_tasks]

b = pass_vec(baseline)
m = pass_vec(mechanism)
a = pass_vec(rule_judge)

def mc(x, y):
    tbl = np.array([
        [sum(i == 1 and j == 1 for i, j in zip(x, y)), sum(i == 1 and j == 0 for i, j in zip(x, y))],
        [sum(i == 0 and j == 1 for i, j in zip(x, y)), sum(i == 0 and j == 0 for i, j in zip(x, y))]
    ])
    r = mcnemar(tbl, exact=True)
    d = sum(y)/len(y) - sum(x)/len(x)
    return d, r.pvalue

delta_a, p_a = mc(b, m)
delta_b, p_b = mc(a, m)

out = {
    "algorithm": "ORPO",
    "baseline": {"pass_rate": sum(b)/len(b), "n": len(b)},
    "mechanism": {"pass_rate": sum(m)/len(m), "n": len(m)},
    "ablation": {"pass_rate": sum(a)/len(a), "n": len(a)},
    "delta_a": delta_a, "p_a": p_a,
    "delta_b": delta_b, "p_b": p_b
}
Path("/content/ablation_results_orpo.json").write_text(json.dumps(out, indent=2))
print("Saved /content/ablation_results_orpo.json")
